# DPO Rewriter Data Generation
Generates `data/dpo/{train,val}.jsonl` — preference pairs `(chosen_rewrite, rejected_rewrite)` per `(question, persona)` used to train the DPO query rewriter.

**Approach (proxy-judge, remote rewriter):** For each `(question, persona)`, sample N=4 rewrites at varying temperatures from a remote Grok model (`grok-4-1-fast`); call a separate LLM judge once per rewrite to rate how well it would help retrieve the right study material for that learner; pair the highest- and lowest-scoring rewrites as `(chosen, rejected)`. Cross-persona negatives (scholar's best → crammer's rejected) are added for free. All rewriter and judge calls within a question are parallelised with `ThreadPoolExecutor(max_workers=4)`.

> **Why remote rewriter?** An earlier version used a local Gemma-4-E4B via Unsloth. The quality of generated rewrites was insufficient, so the rewriter was replaced with a remote Grok model. The judge remains a separate model family (OpenAI-compatible) to preserve judge independence.

**Kaggle setup checklist**
1. Internet access must be enabled.
2. Add Kaggle secrets: `OPENAI_API_KEY` and `OPENAI_BASE_URL` (judge), `REWRITER_API_KEY` and `REWRITER_BASE_URL` (rewriter/Grok).
3. Attach the `simurgh-data` dataset (contains `questions/` and `splits/`).
4. No GPU required — both models are API calls.

In [ ]:
!pip install -q openai tqdm

## Config

In [ ]:
import os

# ── Kaggle secrets ────────────────────────────────────────────────────────
try:
    from kaggle_secrets import UserSecretsClient

    _s = UserSecretsClient()
    OPENAI_API_KEY = _s.get_secret("OPENAI_API_KEY")
    OPENAI_BASE_URL = _s.get_secret("OPENAI_BASE_URL")
    REWRITER_API_KEY = _s.get_secret("REWRITER_API_KEY")
    REWRITER_BASE_URL = _s.get_secret("REWRITER_BASE_URL")
except Exception:
    OPENAI_API_KEY = os.environ["OPENAI_API_KEY"]
    OPENAI_BASE_URL = os.environ.get("OPENAI_BASE_URL")
    REWRITER_API_KEY = os.environ.get("REWRITER_API_KEY", "")
    REWRITER_BASE_URL = os.environ.get("REWRITER_BASE_URL")

# ── Paths ────────────────────────────────────────────────────────
DATASET_SLUG = "simurgh-data"
DATA_ROOT = f"/kaggle/input/datasets/alirezahsn/{DATASET_SLUG}"
OUTPUT_DIR = "/kaggle/working/data/dpo"

# ── Inline config (mirrors configs/datagen_dpo.yaml) ───────────────────────
CFG = {
    "rewriter": {
        "model": "grok-4-1-fast",
        "temperatures": [0.1, 0.4, 0.8, 1.1],
        "max_completion_tokens": 300,
        "max_workers": 4,
    },
    "judge": {
        "model": "gpt-4.1-nano",
        "temperature": 0.0,
        "max_completion_tokens": 16,
    },
    "cross_persona_threshold": 0.2,
    "data": {
        "questions_dir": f"{DATA_ROOT}/questions",
        "splits_dir": f"{DATA_ROOT}/splits",
        "output_dir": OUTPUT_DIR,
    },
    "seed": 42,
}


## Core classes

In [ ]:
import openai


class OpenAICompatClient:
    def __init__(self, base_url, api_key, model, temperature=0.0, max_completion_tokens=16):
        self.client = openai.OpenAI(base_url=base_url, api_key=api_key)
        self.model = model
        self.temperature = temperature
        self.max_completion_tokens = max_completion_tokens

    def chat(self, messages: list) -> str:
        resp = self.client.chat.completions.create(
            model=self.model,
            messages=messages,
            temperature=self.temperature,
            max_completion_tokens=self.max_completion_tokens,
        )
        return resp.choices[0].message.content or ""

In [ ]:
from dataclasses import dataclass


@dataclass(frozen=True)
class Profile:
    id: str
    split: str
    rendered: str


PERSONAS = {
    "crammer": Profile(
        id="crammer",
        split="train",
        rendered=(
            "A ninth-grader who finds the textbook hard to follow and has little background "
            "on this topic. Mainly wants to pass the exam — give the answer and what is needed "
            "to score — but it must be spelled out simply, step by step, with examples."
        ),
    ),
    "scholar": Profile(
        id="scholar",
        split="train",
        rendered=(
            "A ninth-grader who reads dense material easily and has solid background on this "
            "topic. Wants to understand the underlying why and how, and the connections between "
            "ideas. Prefers a terse, high-level treatment without hand-holding or padding."
        ),
    ),
    "steady": Profile(
        id="steady",
        split="train",
        rendered=(
            "A capable ninth-grader with average background on this topic. Wants a correct "
            "answer with a brief justification, balanced toward exam needs. Does not need "
            "elaborate scaffolding, but does appreciate a one-line reason."
        ),
    ),
}


def render_profile(persona_id: str) -> str:
    return PERSONAS[persona_id].rendered


def train_personas() -> list:
    return [p for p in PERSONAS.values() if p.split == "train"]

In [ ]:
_REWRITE_SYSTEM = (
    "You are a query rewriting assistant for a Persian educational RAG system. "
    "Given a learner profile and an original question, rewrite the question as a "
    "retrieval query that will surface the most pedagogically useful passages for "
    "that specific learner. "
    "Rules: output ONLY the rewritten query — no explanation, no preamble, no quotes. "
    "Keep it in Persian if the original is Persian. "
    "You may expand abbreviations, add prerequisite terms, or rephrase for clarity, "
    "but do not invent facts or change the question's intent."
)


def build_rewrite_prompt(profile_rendered: str, query: str) -> list:
    user = (
        f"Learner profile: {profile_rendered}\n\n"
        f"Original question: {query}\n\n"
        "Rewritten retrieval query:"
    )
    return [
        {"role": "system", "content": _REWRITE_SYSTEM},
        {"role": "user", "content": user},
    ]

## Helper functions

In [ ]:
import json
import logging
import re
from pathlib import Path

from tqdm.auto import tqdm

logging.basicConfig(
    level=logging.INFO,
    format="%(asctime)s [%(levelname)s] %(message)s",
    datefmt="%H:%M:%S",
)
logger = logging.getLogger(__name__)

FLOAT_RE = re.compile(r"[\d.]+")

DPO_JUDGE_SYSTEM = (
    "You are an expert Persian language tutor evaluating query rewrites "
    "for a RAG retrieval system."
)


def build_judge_messages(persona_rendered: str, original_query: str, rewrite: str) -> list:
    user = (
        "A student with the following profile is searching for study material:\n"
        f"Profile: {persona_rendered}\n\n"
        f"Original exam question: {original_query}\n\n"
        f"Candidate rewrite: {rewrite}\n\n"
        "Rate 0.0\u20131.0 how well this rewrite would help retrieve the right study material "
        "for this specific student. A good rewrite should:\n"
        "  - Preserve the original question's meaning\n"
        "  - Use vocabulary and framing that matches the student's profile\n"
        "  - Be specific enough to surface relevant passages at the right depth\n\n"
        "Respond with a single decimal number only, e.g. 0.61"
    )
    return [
        {"role": "system", "content": DPO_JUDGE_SYSTEM},
        {"role": "user", "content": user},
    ]


def parse_score(response: str) -> float:
    m = FLOAT_RE.search(response.strip())
    if m is None:
        logger.warning("Could not parse score from: %r", response)
        return 0.0
    return max(0.0, min(1.0, float(m.group())))


def index_questions(questions_dir: Path) -> dict:
    result = {}
    for fpath in sorted(questions_dir.glob("*.json")):
        result[fpath.stem] = fpath
    logger.info("Found %d question files in %s", len(result), questions_dir)
    return result


def load_question_stem(exam_file: Path, qid: str) -> str:
    data = json.loads(exam_file.read_text(encoding="utf-8"))
    for q in data.get("questions", []):
        if q.get("id") == qid:
            return q["stem"]
    raise KeyError(f"Question {qid!r} not found in {exam_file}")


def read_split_qids(splits_dir: Path, split_name: str) -> list:
    split_file = splits_dir / f"{split_name}_qids.txt"
    if not split_file.exists():
        raise FileNotFoundError(f"Split file not found: {split_file}")
    pairs = []
    for line in split_file.read_text(encoding="utf-8").splitlines():
        line = line.strip()
        if not line or line.startswith("#") or ":" not in line:
            continue
        exam_stem, qid = line.split(":", 1)
        pairs.append((exam_stem.strip(), qid.strip()))
    return pairs

## Run pipeline

In [ ]:
# Health check: 4 rewrites + judge scores for one sample query.
# Runs before the full pipeline to confirm both models are working.

SAMPLE_QUERY = "انرژی جنبشی یک جسم با جرم ۲ کیلوگرم که با سرعت ۳ متر بر ثانیه حرکت می‌کند چقدر است؟"
CHECK_PERSONA = "crammer"

check_profile = PERSONAS[CHECK_PERSONA].rendered
_check_judge = OpenAICompatClient(
    base_url=OPENAI_BASE_URL,
    api_key=OPENAI_API_KEY,
    model=CFG["judge"]["model"],
    temperature=CFG["judge"]["temperature"],
    max_completion_tokens=CFG["judge"]["max_completion_tokens"],
)

print(f"Persona : {CHECK_PERSONA}")
print(f"Query   : {SAMPLE_QUERY}\n")

for temp in CFG["rewriter"]["temperatures"]:
    _rw_client = OpenAICompatClient(
        base_url=REWRITER_BASE_URL,
        api_key=REWRITER_API_KEY,
        model=CFG["rewriter"]["model"],
        temperature=temp,
        max_completion_tokens=CFG["rewriter"]["max_completion_tokens"],
    )
    msgs = build_rewrite_prompt(check_profile, SAMPLE_QUERY)
    rewrite = _rw_client.chat(msgs)
    judge_msgs = build_judge_messages(check_profile, SAMPLE_QUERY, rewrite)
    raw = _check_judge.chat(judge_msgs)
    m = FLOAT_RE.search(raw.strip())
    score = max(0.0, min(1.0, float(m.group()))) if m else 0.0
    print(f"  temp={temp:.1f}  score={score:.2f}  {rewrite[:100]}")

print("\nHealth check complete.")


In [ ]:
import random
from collections import defaultdict
from concurrent.futures import ThreadPoolExecutor, as_completed

from tqdm.auto import tqdm

random.seed(CFG["seed"])

questions_dir = Path(CFG["data"]["questions_dir"])
splits_dir = Path(CFG["data"]["splits_dir"])
output_dir = Path(CFG["data"]["output_dir"])
output_dir.mkdir(parents=True, exist_ok=True)

temperatures = CFG["rewriter"]["temperatures"]
max_workers = CFG["rewriter"]["max_workers"]
cross_threshold = CFG["cross_persona_threshold"]
train_profiles = train_personas()
persona_ids = [p.id for p in train_profiles]

questions_map = index_questions(questions_dir)

# One OpenAICompatClient per temperature (same Grok model, different temp).
rewriter_clients = {
    temp: OpenAICompatClient(
        base_url=REWRITER_BASE_URL,
        api_key=REWRITER_API_KEY,
        model=CFG["rewriter"]["model"],
        temperature=temp,
        max_completion_tokens=CFG["rewriter"]["max_completion_tokens"],
    )
    for temp in temperatures
}

judge = OpenAICompatClient(
    base_url=OPENAI_BASE_URL,
    api_key=OPENAI_API_KEY,
    model=CFG["judge"]["model"],
    temperature=CFG["judge"]["temperature"],
    max_completion_tokens=CFG["judge"]["max_completion_tokens"],
)


def _run_one_combo(persona_id, temp, rw_client, judge_client, persona_rendered, query):
    try:
        msgs = build_rewrite_prompt(persona_rendered, query)
        rewrite = rw_client.chat(msgs).strip()
    except Exception as exc:
        logger.warning("Rewrite failed persona=%s temp=%.1f: %s", persona_id, temp, exc)
        return persona_id, None, 0.0
    try:
        jmsgs = build_judge_messages(persona_rendered, query, rewrite)
        raw = judge_client.chat(jmsgs)
        m = FLOAT_RE.search(raw.strip())
        score = max(0.0, min(1.0, float(m.group()))) if m else 0.0
    except Exception as exc:
        logger.warning("Judge failed persona=%s temp=%.1f: %s", persona_id, temp, exc)
        score = 0.0
    return persona_id, rewrite, score


for split_name in ("train", "val"):
    try:
        qid_pairs = read_split_qids(splits_dir, split_name)
    except FileNotFoundError as e:
        logger.warning("%s — skipping %s split", e, split_name)
        continue

    output_path = output_dir / f"{split_name}.jsonl"
    records_written = 0
    logger.info(
        "Processing %s split (%d questions) → %s", split_name, len(qid_pairs), output_path
    )

    with output_path.open("w", encoding="utf-8") as fh:
        for exam_stem, qid in tqdm(qid_pairs, desc=split_name, unit="q"):
            exam_file = questions_map.get(exam_stem)
            if exam_file is None:
                logger.warning("Exam file not found for stem %r", exam_stem)
                continue
            try:
                query = load_question_stem(exam_file, qid)
            except (KeyError, json.JSONDecodeError) as e:
                logger.warning("Skipping %s:%s: %s", exam_stem, qid, e)
                continue

            combos = [(pid, temp) for pid in persona_ids for temp in temperatures]
            candidates_by_persona = defaultdict(list)

            with ThreadPoolExecutor(max_workers=max_workers) as executor:
                future_map = {
                    executor.submit(
                        _run_one_combo,
                        pid, temp,
                        rewriter_clients[temp],
                        judge,
                        render_profile(pid),
                        query,
                    ): (pid, temp)
                    for pid, temp in combos
                }
                for future in as_completed(future_map):
                    pid, rewrite, score = future.result()
                    if rewrite is not None:
                        candidates_by_persona[pid].append((rewrite, score))

            best_for_persona = {}
            best_score_for_persona = {}

            for persona_id in persona_ids:
                candidates = candidates_by_persona[persona_id]
                if not candidates:
                    logger.error("No candidates for %s:%s persona=%s", exam_stem, qid, persona_id)
                    continue
                candidates.sort(key=lambda x: x[1], reverse=True)
                chosen, chosen_score = candidates[0]
                rejected, _ = candidates[-1]
                best_for_persona[persona_id] = chosen
                best_score_for_persona[persona_id] = chosen_score
                fh.write(json.dumps({"persona_id": persona_id, "query": query, "chosen": chosen, "rejected": rejected}, ensure_ascii=False) + "\n")
                records_written += 1

            for i, pid_a in enumerate(persona_ids):
                if pid_a not in best_for_persona:
                    continue
                for pid_b in persona_ids[i + 1:]:
                    if pid_b not in best_for_persona:
                        continue
                    if abs(best_score_for_persona[pid_a] - best_score_for_persona[pid_b]) <= cross_threshold:
                        continue
                    fh.write(json.dumps({"persona_id": pid_a, "query": query, "chosen": best_for_persona[pid_a], "rejected": best_for_persona[pid_b]}, ensure_ascii=False) + "\n")
                    records_written += 1
                    fh.write(json.dumps({"persona_id": pid_b, "query": query, "chosen": best_for_persona[pid_b], "rejected": best_for_persona[pid_a]}, ensure_ascii=False) + "\n")
                    records_written += 1

    logger.info("Wrote %d records to %s", records_written, output_path)
